# 01 — LLM Failure Taxonomy

Understanding failure modes is prerequisite to building reliable LLM systems.

## Hallucination Types: Factual, Reasoning, Faithfulness

LLM failures can be categorised along several axes:

**Hallucination types**:
- *Factual hallucination*: generating plausible-sounding but incorrect facts — wrong dates, nonexistent citations, false statistics
- *Reasoning hallucination*: correct premises, incorrect conclusions — the model 'reasons' to a wrong answer
- *Faithfulness hallucination*: given a source document, the summary or answer contradicts it
- *Attribution hallucination*: correctly retrieved fact but misattributed to the wrong source

**Error categories by source**:
- *Knowledge cutoff*: model doesn't know events after training data
- *Long-tail knowledge*: rare entities and obscure facts are underrepresented in training
- *Sycophancy*: model agrees with the user's (incorrect) framing
- *Overthinking / second-guessing*: correct initial answer, then incorrectly revised
- *Format forcing*: user demands a specific format and model sacrifices accuracy for compliance

**Why hallucination is hard to fix**: LLMs are trained to be fluent and confident — they don't have a 'don't know' mode by default. Calibration and uncertainty quantification must be added explicitly.

In [ ]:
# Failure mode classifier on LLM outputs
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Simulate an LLM output classifier
# In production: use NLI model or GPT-4 judge
examples = [
    ('The Eiffel Tower was built in 1850.', 'factual_hallucination'),
    ('Paris is the capital of France.', 'correct'),
    ('All prime numbers are odd, so 2 is not prime.', 'reasoning_error'),
    ('According to the document, X said Y.', 'correct'),
    ('Studies show 73% of people prefer Python.', 'factual_hallucination'),
    ('If A implies B and B implies C, then A implies C.', 'correct'),
    ('The paper by Smith (2019) found that...', 'attribution_error'),
    ('The quantum tunneling effect explains how you can teleport.', 'reasoning_error'),
    ('Water freezes at 0 degrees Celsius at sea level.', 'correct'),
    ('Einstein won the Nobel Prize for relativity.', 'factual_hallucination'),
    ('Python was created by Guido van Rossum.', 'correct'),
    ('Since more data is always better, overfitting is not a concern.', 'reasoning_error'),
    ('The document states climate change is not real, which is correct.', 'faithfulness_error'),
    ('The transformer architecture uses attention mechanisms.', 'correct'),
    ('Newton invented calculus alone, without Leibniz.', 'factual_hallucination'),
]

texts, labels = zip(*examples)
label_map = {l: i for i, l in enumerate(set(labels))}
y = [label_map[l] for l in labels]

# Add more synthetic examples
np.random.seed(42)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=200)
X_feat = vectorizer.fit_transform(texts).toarray()

clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_feat, y)

print('Failure mode categories:')
for label, idx in sorted(label_map.items()):
    count = sum(1 for l in labels if l == label)
    print(f'  {label}: {count} examples')

# Classify new outputs
inv_map = {v: k for k, v in label_map.items()}
test_outputs = [
    'The sun orbits the earth.',
    'Gradient descent minimises the loss function.',
    'As I recall, the paper proved that X equals 42.',
]

print('\nClassified outputs:')
for text in test_outputs:
    feat = vectorizer.transform([text]).toarray()
    pred = inv_map[clf.predict(feat)[0]]
    proba = clf.predict_proba(feat).max()
    print(f'  [{pred} ({proba:.2f})]: "{text[:60]}"')